In [ ]:
import os
from tqdm import tqdm 

from pathlib import Path
import lightning as L
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import scvi
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
import torch
from torch.utils.data import TensorDataset, DataLoader
from torchdyn.core import NeuralODE
from tqdm.auto import tqdm 
from pathlib import Path 
from scRatio.models.flow_matching import ConditionalFlowMatchingWithScore
from scRatio.models.node_wrappers import div_fn_hutch_trace

import sys
sys.path.insert(0, "../../runs/differential_abundance/")
from utils_scratio import train

In this experiment, we will use generative models for individual distribution (conditioned) and a model trained on all the cells (unconditional) as a simulation field.

In [27]:
class NODEWrapper_with_ratio_generic_models(torch.nn.Module):
    def __init__(self, model_num, model_den, model_vf):
        super().__init__()
        self.model_den = model_den
        self.model_num = model_num
        self.model_vf = model_vf
        self.div_fn, self.eps_fn = div_fn_hutch_trace, torch.randn_like

    def forward(self, t, x, *args, **kwargs):
        x = x[..., :-1]
        
        def vecfield(y):
            ut, _ = self.model_num(y.unsqueeze(0), t, None, use_conds=[False])
            vt, _ = self.model_den(y.unsqueeze(0), t, None, use_conds=[False])
            return vt.squeeze() - ut.squeeze()
            
        div = torch.vmap(self.div_fn(vecfield))(x, self.eps_fn(x))
        
        ut, score_u = self.model_num(x, t, None, use_conds=[False])
        vt, score_v = self.model_den(x, t, None, use_conds=[False])
        ft, _ = self.model_vf(x, t, None, use_conds=[False])
        
        correction_term_u = torch.linalg.vecdot(ft - ut, score_u)
        correction_term_v = torch.linalg.vecdot(vt - ft, score_v)
        dr = div + correction_term_u + correction_term_v
        
        return torch.cat([ft, dr[:, None]], dim=-1)

def compute_ratio(data_samples, model_num, model_den, model_vf, batch_size):
    # Initialize the device 
    device = data_samples.device   

    # Initialize torch dataloader to iterate through the samples 
    dataloader = DataLoader(TensorDataset(data_samples), batch_size=batch_size, drop_last=False)
    log_ratios = []
    for batch in tqdm(dataloader):
        X_batch = batch[0]
        
        # correction term - its own field
        node = NeuralODE(
            NODEWrapper_with_ratio_generic_models(model_num,
                                                  model_den,
                                                  model_vf),
            solver="dopri5", sensitivity="adjoint", atol=1e-4, rtol=1e-4
        )
        
        with torch.no_grad():
            traj = node.trajectory(
                torch.cat([X_batch, torch.zeros(X_batch.shape[0], 1).to(device)], dim=-1),
                t_span=torch.linspace(1, 0, 100).to(device)
            )
        z0, ratio = traj[-1][:, :-1], traj[-1][:, -1]
        log_ratio_hat = -ratio.cpu().numpy()
        log_ratios.append(log_ratio_hat)

    return np.concatenate(log_ratios)

# Initialize flow model 

In [ ]:
X_block = np.load("../../project_folder/data/mi_data/block_sigma_320.npy")
X_prior = np.load("../../project_folder/data/mi_data/identity_sigma_320.npy")
print(X_block.shape)
print(X_prior.shape)

X_block_train, X_prior_train =  X_block[:-10000], X_prior[:-10000] 
X_block_test, X_prior_test = X_block[90000:], X_prior[90000:] 
print(X_block_train.shape)
print(X_prior_train.shape)
print(X_block_test.shape)
print(X_prior_test.shape)

X_block_train, X_prior_train =  torch.from_numpy(X_block_train).to("cuda"), torch.from_numpy(X_prior_train).to("cuda")
X_block_test, X_prior_test = torch.from_numpy(X_block_test).to("cuda"), torch.from_numpy(X_prior_test).to("cuda")

(100000, 320)
(100000, 320)
(90000, 320)
(90000, 320)
(10000, 320)
(10000, 320)


# Train separate flows 

In [10]:
# Train a very deterministic flow 
sigma = 0.25
sigma_min = 0
lambda_t = lambda t: torch.sqrt((1 - (1 - sigma_min) * t) ** 2 + sigma * t * (1 - t))
lambda_sp_t = lambda t: (sigma * (1 - 2 * t) - 2 * (1 - sigma_min) * (1 - (1 - sigma_min) * t)) / 2
n_steps = 40_000
batch_size = 512

**Prior**

In [11]:
model_prior = ConditionalFlowMatchingWithScore(input_dim=320,
                                                cond_dims=[0],
                                                hidden_dims=[1024, 1024, 1024],
                                                encoder_hidden_dims=[256],
                                                encoder_out_dim=256,
                                                encoder_out_dim_cond=50,
                                                time_feature_dim=50,
                                                lambda_t=lambda_t,
                                                lambda_sp_t=lambda_sp_t,
                                                betas=[1], 
                                                lr=1e-4, 
                                                dropout = 0
                                            ).to("cuda")

optimizer = model_prior.configure_optimizers()
print(model_prior)

ConditionalFlowMatchingWithScore(
  (data_encoder): Encoder(
    (encoder): Sequential(
      (0): Linear(in_features=320, out_features=256, bias=True)
      (1): SELU()
      (2): Dropout(p=0, inplace=False)
      (3): Linear(in_features=256, out_features=256, bias=True)
    )
  )
  (cond_encoders): ModuleList(
    (0): Encoder(
      (encoder): Sequential(
        (0): Linear(in_features=0, out_features=256, bias=True)
        (1): SELU()
        (2): Dropout(p=0, inplace=False)
        (3): Linear(in_features=256, out_features=50, bias=True)
      )
    )
  )
  (vf_mlp): FlowMatchingMLP(
    (mlp): Sequential(
      (0): Linear(in_features=356, out_features=1024, bias=True)
      (1): SELU()
      (2): Dropout(p=0, inplace=False)
      (3): Linear(in_features=1024, out_features=1024, bias=True)
      (4): SELU()
      (5): Dropout(p=0, inplace=False)
      (6): Linear(in_features=1024, out_features=1024, bias=True)
      (7): SELU()
      (8): Dropout(p=0, inplace=False)
      (9): 

In [14]:
model_prior = train(batch_size, 
                    n_steps, 
                    model_prior,
                    optimizer, 
                    X_prior_train,
                    None)

In [ ]:
plt.figure(figsize=(3,3))
plt.plot(model_prior.vf_losses)
plt.plot(model_prior.score_losses)

**Block diag**

In [15]:
model_block = ConditionalFlowMatchingWithScore(input_dim=320,
                                                cond_dims=[0],
                                                hidden_dims=[1024, 1024, 1024],
                                                encoder_hidden_dims=[256],
                                                encoder_out_dim=256,
                                                encoder_out_dim_cond=50,
                                                time_feature_dim=50,
                                                lambda_t=lambda_t,
                                                lambda_sp_t=lambda_sp_t,
                                                betas=[1], 
                                                lr=1e-4, 
                                                dropout = 0
                                            ).to("cuda")

optimizer = model_block.configure_optimizers()
print(model_block)

ConditionalFlowMatchingWithScore(
  (data_encoder): Encoder(
    (encoder): Sequential(
      (0): Linear(in_features=320, out_features=256, bias=True)
      (1): SELU()
      (2): Dropout(p=0, inplace=False)
      (3): Linear(in_features=256, out_features=256, bias=True)
    )
  )
  (cond_encoders): ModuleList(
    (0): Encoder(
      (encoder): Sequential(
        (0): Linear(in_features=0, out_features=256, bias=True)
        (1): SELU()
        (2): Dropout(p=0, inplace=False)
        (3): Linear(in_features=256, out_features=50, bias=True)
      )
    )
  )
  (vf_mlp): FlowMatchingMLP(
    (mlp): Sequential(
      (0): Linear(in_features=356, out_features=1024, bias=True)
      (1): SELU()
      (2): Dropout(p=0, inplace=False)
      (3): Linear(in_features=1024, out_features=1024, bias=True)
      (4): SELU()
      (5): Dropout(p=0, inplace=False)
      (6): Linear(in_features=1024, out_features=1024, bias=True)
      (7): SELU()
      (8): Dropout(p=0, inplace=False)
      (9): 

In [17]:
model_block = train(batch_size,
                    n_steps, 
                    model_block, 
                    optimizer, 
                    X_block_train,
                    None)

In [ ]:
plt.figure(figsize=(3,3))
plt.plot(model_block.vf_losses)
plt.plot(model_block.score_losses)

**Unconditional**

In [18]:
model_unconditional = ConditionalFlowMatchingWithScore(input_dim=320,
                                                        cond_dims=[0],
                                                        hidden_dims=[1024, 1024, 1024],
                                                        encoder_hidden_dims=[256],
                                                        encoder_out_dim=256,
                                                        encoder_out_dim_cond=50,
                                                        time_feature_dim=50,
                                                        lambda_t=lambda_t,
                                                        lambda_sp_t=lambda_sp_t,
                                                        betas=[1], 
                                                        lr=1e-4, 
                                                        dropout = 0
                                                    ).to("cuda")

optimizer = model_unconditional.configure_optimizers()
print(model_unconditional)

ConditionalFlowMatchingWithScore(
  (data_encoder): Encoder(
    (encoder): Sequential(
      (0): Linear(in_features=320, out_features=256, bias=True)
      (1): SELU()
      (2): Dropout(p=0, inplace=False)
      (3): Linear(in_features=256, out_features=256, bias=True)
    )
  )
  (cond_encoders): ModuleList(
    (0): Encoder(
      (encoder): Sequential(
        (0): Linear(in_features=0, out_features=256, bias=True)
        (1): SELU()
        (2): Dropout(p=0, inplace=False)
        (3): Linear(in_features=256, out_features=50, bias=True)
      )
    )
  )
  (vf_mlp): FlowMatchingMLP(
    (mlp): Sequential(
      (0): Linear(in_features=356, out_features=1024, bias=True)
      (1): SELU()
      (2): Dropout(p=0, inplace=False)
      (3): Linear(in_features=1024, out_features=1024, bias=True)
      (4): SELU()
      (5): Dropout(p=0, inplace=False)
      (6): Linear(in_features=1024, out_features=1024, bias=True)
      (7): SELU()
      (8): Dropout(p=0, inplace=False)
      (9): 

In [ ]:
model_unconditional = train(batch_size, 
                            n_steps,
                            model_unconditional,
                            optimizer, 
                            torch.cat([X_block_train, X_prior_train], dim=0), None)

## Try on the leavout datapoints

In [ ]:
ratio_estimations = compute_ratio(X_block_test,
                                  model_block, 
                                  model_prior, 
                                  model_unconditional, 
                                  1000)

In [ ]:
plt.hist(ratio_estimations)

In [ ]:
ratio_estimations.mean()